# Jealous Couples

Three couples need to cross a river.  They have to follow the following rules: 
- The boat can only carry two persons.
- At least one person has to be on the boat on every crossing.
- The husbands know their wives well and would never leave them with another man, unless they are present themselves.

We are going to formulate this problem as a *symbolic transition system*.
Then, we can solve the problem using the constraint solver `Z3`.

We assume the first couple is called `Anton` and `Doris`, the names of the second couple are
`Brian` and `Eliza`, while the last couple is called `Charly` and `Freya`.  Hence we
will use the following variables to encode the problem:
* `A` equals `1` if `Anton` is on the western shore,
* `B` equals `1` if `Brian` is on the western shore,
* `C` equals `1` if `Charly` is on the western shore,
* `D` equals `1` if `Doris` is on the western shore,
* `E` equals `1` if `Eliza` is on the western shore,
* `F` equals `1` if `Freya` is on the western shore,
* `S` equals `1` if the ship is on the western shore.

In [ ]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

In [ ]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

In [ ]:
let A = Int "A"
let B = Int "B"
let C = Int "C"
let D = Int "D"
let E = Int "E"
let F = Int "F"
let S = Int "S"

The function `start` takes the seven variables describing a state.  It returns a list of constraints that describe the initial state.

In [ ]:
let start (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    failwith "your code here"

In [ ]:
start A B C D E F S

The function `goal` takes the seven variables describing a state.  It returns a list of constraints that describe the final state.

In [ ]:
let goal (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    failwith "your code here"

In [ ]:
goal A B C D E F S

The function `invariant` takes the seven variables describing a state.  It returns a list of formulas that need to be true after every transition:  If a wife is not on the same shore as her husband, then she must not be on the same shore as one of the other husbands.

In [ ]:
let invariant (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    failwith "your code here"

In [ ]:
invariant A B C D E F S

The function `transition` takes fourteen arguments:  the variables `A`, $\cdots$, `S` describe the state before the crossing, while `Ax`, $\cdots$, `Sx` describe the state after the crossing.  It returns a list of formulas that describe the transitions:
- If the ship is on the western shore, one or two persons travel from the western shore to the eastern shore.
- If the ship is on the eastern shore, one or two persons travel from the eastern shore to the western shore.
- The ship changes sides.

In [ ]:
let transition (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr)
               (Ax: IntExpr) (Bx: IntExpr) (Cx: IntExpr) (Dx: IntExpr) (Ex: IntExpr) (Fx: IntExpr) (Sx: IntExpr)
               : BoolExpr list =
    failwith "your code here"

The function `jealousCSP` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem with `n` crossings.  If this is possible, `Some Solution` is returned, where `Solution` is a map assigning values to the variables.  Otherwise, `None` is returned.

In [ ]:
let jealousCSP (n: int) : Map<string, int> option =
    let Solver = newSolver ()
    let vars name = [| for i in 0 .. n -> Int (sprintf "%s%d" name i) |]
    let As, Bs, Cs, Ds, Es, Fs, Ss = vars "A", vars "B", vars "C", vars "D", vars "E", vars "F", vars "S"
    let Cts = [ yield! start As[0] Bs[0] Cs[0] Ds[0] Es[0] Fs[0] Ss[0]
                yield! goal  As[n] Bs[n] Cs[n] Ds[n] Es[n] Fs[n] Ss[n]
                for i in 0 .. n - 1 do
                    let j = i + 1
                    yield! invariant  As[i] Bs[i] Cs[i] Ds[i] Es[i] Fs[i] Ss[i]
                    yield! transition As[i] Bs[i] Cs[i] Ds[i] Es[i] Fs[i] Ss[i]
                                      As[j] Bs[j] Cs[j] Ds[j] Es[j] Fs[j] Ss[j]
                    for X in [ As; Bs; Cs; Ds; Es; Fs; Ss ] do
                        yield 0 .<= X[i]
                        yield X[i] .<= 1 ]
    Solver.Add(Array.ofList Cts)
    if Solver.Check() = Status.SATISFIABLE then
        let m = Solver.Model
        Some (Map [ for (name, X) in [ ("A", As); ("B", Bs); ("C", Cs); ("D", Ds); ("E", Es); ("F", Fs) ] do
                      for i in 0 .. n -> (sprintf "%s%d" name i, intValue m X[i]) ])
    else
        None

In [ ]:
jealousCSP 3

The function `findSolution` takes no arguments.  It tries $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings together with the solution.

In [ ]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match jealousCSP n with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [ ]:
#!time
let n, Solution = findSolution ()

In [ ]:
n, Solution

## Auxiliary Code for Pretty Printing

The function `showSolution` takes two arguments:
- `Solution` is a map assigning values to the variables,
- `n` is the number of crossings.

It prints the path that has been found.  We won't discuss the details of this function.

In [ ]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    let get name i = Solution[sprintf "%s%d" name i]
    let persons A B C D E F = rep A "👨" + rep B "👨🏼" + rep C "👨🏿" + rep D "👩" + rep E "👩🏼" + rep F "👩🏿"
    for i in 0 .. n do
        let A, B, C, D, E, F = get "A" i, get "B" i, get "C" i, get "D" i, get "E" i, get "F" i
        printfn "%s%s%s" (persons A B C D E F) (String.replicate 42 " ") (persons (1-A) (1-B) (1-C) (1-D) (1-E) (1-F))
        if i < n then
            let sign = if i % 2 = 0 then 1 else -1
            let crossing name = sign * (get name i - get name (i+1))
            let arrows = if i % 2 = 0 then ">>>" else "<<<"
            printfn "%s%s%s%s" (String.replicate 24 " ") arrows
                    (persons (crossing "A") (crossing "B") (crossing "C") (crossing "D") (crossing "E") (crossing "F")) arrows

In [ ]:
showSolution Solution n